# Step 13 (notebook): one continuous exploratory scan

Same analysis as `13_continuous_exploratory_scan.py`, but the scan is set below instead of in the terminal, and the results are shown here.

1. Set `SCAN` to a folder in `data/continuous-exploratory-raw/` (its `.mat` files, or a zip of them), **or** set `DATA_PATH` to a folder anywhere else (local or NAS; on macOS a connected NAS is under `/Volumes/<share>/...`, and an `smb://` address is translated to that), which is linked into `data/continuous-exploratory-raw/`.
2. Run all cells with the **t-difc-env** kernel (the project's environment, built from `requirements.txt`: Python 3.11, TensorFlow 2.13). In VS Code: *Select Kernel* (top right) > *Jupyter Kernel…* > `t-difc-env (Python 3.11)`. A kernel with TensorFlow ≥ 2.16 / Keras 3 cannot load the saved models and stops in the setup cell.

The scan is analysed with `independent_scan.analyse(..., pulsed=False)`, exactly as step 13 does, so the files written are the same as the script's. A continuous recording is not pulsed: every file is preprocessed as one laser-ON segment. The scan is never grouped with the clearance cohort or with other scans. The `.mat` files are only read; results go to `results/continuous-exploratory/<scan>/`, caches to `data/continuous-exploratory-preprocessed/` and `data/continuous-exploratory-als-cache/`. Rerunning uses the caches.

After the analysis, the plotting cells only read the files written, so they can be rerun on their own (change a setting, rerun the cell).

In [ ]:
# ---- settings -------------------------------------------------------------------
SCAN = "MC38-C57-M5_102726"   # scan folder in data/continuous-exploratory-raw/
DATA_PATH = None              # None = use SCAN above; or the path of a folder with the scan's .mat files,
                              # linked in under the name SCAN (SCAN = None: the folder name of DATA_PATH)

# candidate windows (last cells: detected peaks, artifacts)
SORT_BY = "time"         # "time" or "probability" (highest first)
TIME_RANGE_H = None      # None = whole recording; e.g. (2.0, 2.5) = only candidates between 2.0 h and 2.5 h
MAX_WINDOWS = 48         # None = all of them (a continuous scan can have thousands); a number = the first ones after sorting (per cell)
WINDOWS_PER_PAGE = 24    # windows per figure (4 per row)

## Setup

In [ ]:
import importlib
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

HERE = os.path.abspath("")
SCRIPTS = HERE if os.path.exists(os.path.join(HERE, "common.py")) else os.path.join(HERE, "scripts")
sys.path.insert(0, SCRIPTS)
import common as mc
import independent_scan
import scan_figures as sf
step13 = importlib.import_module("13_continuous_exploratory_scan")   # for its mouse_label (C57-M5 from the file name)

%matplotlib inline
pd.set_option("display.width", 200)
print("python: ", sys.executable)
print("project:", mc.ROOT)
print("cut-off:", mc.CUT)

## The scan

The `.mat` files of the scan and the cell line / mouse label read from their names (they appear in the figures).

In [ ]:
if DATA_PATH:
    SCAN = independent_scan.link_scan(DATA_PATH, mc.CONTINUOUS_RAW_DIR, SCAN)
folder = os.path.join(mc.CONTINUOUS_RAW_DIR, SCAN)
if not os.path.isdir(folder):
    zips = [z for z in os.listdir(mc.CONTINUOUS_RAW_DIR) if z.endswith(".zip")]
    if not zips:
        raise SystemExit(f"no folder {folder}; put the scan's .mat files there, or set DATA_PATH")
    print(f"{folder} not there yet; the analysis unpacks {zips} first")
else:
    mats = sorted(f for f in os.listdir(folder) if f.endswith(".mat") and not f.startswith("."))
    print(f"{SCAN}: {os.path.realpath(folder)}")
    print(f"{len(mats)} .mat file(s): {mats}")
    print("cell line:", sf.cell_line(mats), "| mouse label:", step13.mouse_label(SCAN, mats)[0])
if os.path.isdir(os.path.join(mc.COHORT_RAW_DIR, SCAN)):
    print(f"note: {SCAN} is also a clearance-cohort-raw folder (no cohort result is changed)")

## Run the analysis (as step 13)

`independent_scan.analyse` runs these stages, the same as for the clearance cohort:

1. **Preprocessing** (`full_scan_preprocessing.prepare_scan`, `pulsed=False`): each file is one laser-ON segment; background removal, adaptive peak detection (`parameters.py`) and a 1-s window around every candidate. Cached in `data/continuous-exploratory-preprocessed/<scan>/`, so later runs skip it.
2. **Split-overlap check**: candidates from a file that already has windows in the train/val/test splits are left out of the counts (`split_overlap.csv`, `candidates_in_splits`).
3. **Scoring**: the 5 saved models (seeds) give each candidate a probability; their mean is compared with the cut-off.
4. **Outputs**: `candidates.csv`, timeline, barcode, window galleries, the **clearance fit** of this scan (step 13 only) and `summary.csv`.

The first run of a scan takes a few minutes (preprocessing and scoring); a rerun is faster.

In [ ]:
summary = independent_scan.analyse(SCAN, mc.CONTINUOUS_RAW_DIR, mc.CONTINUOUS_PREP_DIR, mc.CONTINUOUS_ALS_DIR,
                                   mc.CONTINUOUS_DIR, pulsed=False, label=step13.mouse_label)

## Summary

`summary.csv`: recording length, number of candidates, detected peaks (all / not at a laser edge), near misses and peaks per laser-ON hour. (Re)reading it from the results folder lets the cells below run without rerunning the analysis.

In [ ]:
OUT = os.path.join(mc.CONTINUOUS_DIR, SCAN)
pd.read_csv(os.path.join(OUT, "summary.csv")).T.rename(columns={0: "value"})

## Timeline and barcode

Timeline: the barcode above the probability of every candidate (cut-off as a line). Barcode: one line per detected peak, the number of detected peaks on the right.

In [ ]:
for name in ("timeline", "barcode"):
    display(Image(os.path.join(OUT, f"{name}.png"), width=900))

## Clearance fit

Method of step 12, on this scan alone: detected peaks per complete 5-min bin, normalized to the largest bin, time zero = start of the recording.

- Single exponential: $y = e^{-t/\tau}$, half-life $t_{1/2} = \tau \ln 2$
- Double exponential: $y = N_0 e^{-t/\tau_1} + N_1 e^{-t/\tau_2}$ (fast phase $\tau_1$)

No fit is written with fewer than 3 bins or no detections; the single exponential does not converge when the counts do not decay from $t = 0$.

In [ ]:
fit_csv = os.path.join(OUT, "clearance_fit.csv")
if os.path.exists(fit_csv):
    fit = pd.read_csv(fit_csv).iloc[0]
    display(Image(os.path.join(OUT, "clearance_decay.png"), width=600))
    if pd.notna(fit.tau_h):
        print(f"single exponential: tau = {fit.tau_h * 60:.1f} ± {fit.tau_se_h * 60:.1f} min, "
              f"half-life = {fit.half_life_h * 60:.1f} min (reduced chi² {fit.red_chi2_single:.4f})")
    if "tau1_h" in fit and pd.notna(fit.tau1_h):
        print(f"double exponential: N0 = {fit.N0:.2f}, tau1 = {fit.tau1_h * 60:.1f} min, "
              f"N1 = {fit.N1:.2f}, tau2 = {fit.tau2_h * 60:.1f} min (reduced chi² {fit.red_chi2_double:.4f})")
    display(fit.to_frame("value"))
else:
    print("no clearance fit for this scan (see the analysis output above)")

Counts per 5-min bin (`clearance_bins.csv`): `count` = detected peaks in the bin, `normalized` = count / largest bin.

In [ ]:
bins_csv = os.path.join(OUT, "clearance_bins.csv")
bins = pd.read_csv(bins_csv) if os.path.exists(bins_csv) else None
bins

## Peak candidates

`candidates.csv`: every candidate with its time, source file, laser-edge flag, `file_in_splits` and the 5-seed mean probability. `detected` = probability ≥ cut-off, from a file that is not in the train/val/test splits.

In [ ]:
d = pd.read_csv(os.path.join(OUT, "candidates.csv"))
w = np.load(os.path.join(mc.CONTINUOUS_PREP_DIR, SCAN, f"{SCAN}_windows.npz"))
raw, M = w["raw"], w["M"]                     # (N, 2, 2001) raw mV; mask True = laser OFF / zero-padding
assert np.array_equal(w["peaks"], d.peak_index.values)
P_COL = f"p_{mc.MODEL}"
print(f"{len(d)} candidates, {int(d.detected.sum())} detected, "
      f"{int(((d[P_COL] >= mc.NEAR_MISS) & ~d.detected).sum())} near misses ({mc.NEAR_MISS} <= p < {mc.CUT})")
d

Probability of every candidate (5-seed mean): most are close to 0 (artifacts) or 1 (peaks); the ones near the cut-off are the uncertain ones.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(d[P_COL], bins=50, color=sf.PEAK, alpha=0.8)
ax.axvline(mc.CUT, color="k", lw=1, ls="--", label=f"cut-off {mc.CUT}")
ax.set_yscale("log")
ax.set_xlabel("Probability (5-seed mean)")
ax.set_ylabel("Candidates")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 1-s window of every candidate

Raw Ch1 / Ch2 (mV), ±500 ms around the candidate (at 0 ms); grey = samples where both channels are zero (outside the recording, or a zeroed stretch). Each title: candidate number, time, probability, and `edge` / `in splits` when the candidate is within 25 ms of such a grey region (`at_laser_edge`; in a continuous scan there are no laser OFF gaps, so this is the start / end of a file or a zeroed stretch) or comes from a file already in the train/val/test splits.

Which windows are plotted is set by `SORT_BY`, `TIME_RANGE_H` and `MAX_WINDOWS` in the settings cell; change them and rerun the cells below.

In [ ]:
def plot_windows(idx, label, per_page=WINDOWS_PER_PAGE, ncol=4):
    t = (np.arange(raw.shape[-1]) - raw.shape[-1] // 2) / mc.FS * 1e3
    for start in range(0, len(idx), per_page):
        chunk = idx[start:start + per_page]
        nrow = int(np.ceil(len(chunk) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.3 * nrow), squeeze=False)
        for ax, i in zip(axes.ravel(), chunk):
            r = d.iloc[i]
            ax.fill_between(t, 0, 1, where=M[i], transform=ax.get_xaxis_transform(), color="0.85", lw=0)
            ax.plot(t, raw[i, 1], color=sf.ART, lw=0.6, label="Ch2")
            ax.plot(t, raw[i, 0], color=sf.PEAK, lw=0.7, label="Ch1")
            flags = [f for f, on in (("edge", r.at_laser_edge), ("in splits", r.file_in_splits)) if on]
            ax.set_title(f"#{i}  {r.time_h:.3f} h  p = {r[P_COL]:.3f}" + (f"  ({', '.join(flags)})" if flags else ""),
                         fontsize=9, color=sf.PEAK if r.detected else sf.ART)
            ax.set_xlim(t[0], t[-1])
            ax.set_xticks([-500, -250, 0, 250, 500])
            ax.tick_params(labelsize=8)
        for ax in axes.ravel()[len(chunk):]:
            ax.axis("off")
        for ax in axes[:, 0]:
            ax.set_ylabel("Signal (mV)", fontsize=9)
        for ax in axes[-1]:
            ax.set_xlabel("Time (ms)", fontsize=9)
        h, l = axes[0, 0].get_legend_handles_labels()
        fig.legend(h[::-1], l[::-1], loc="upper right", ncol=2, frameon=False, fontsize=9)
        fig.suptitle(f"{SCAN}: {label} {start + 1}-{start + len(chunk)} of {len(idx)}", x=0.01, ha="left",
                     fontsize=10)
        fig.tight_layout()
        plt.show()


def show(mask, label):
    """Plot the candidates where mask is True (within TIME_RANGE_H), sorted by SORT_BY, at most MAX_WINDOWS."""
    mask = np.asarray(mask)
    if TIME_RANGE_H:
        mask = mask & d.time_h.between(*TIME_RANGE_H).values
        label = f"{label} ({TIME_RANGE_H[0]}-{TIME_RANGE_H[1]} h)"
    order = d[mask].sort_values("time_h" if SORT_BY == "time" else P_COL, ascending=SORT_BY == "time").index
    idx = list(order[:MAX_WINDOWS] if MAX_WINDOWS else order)
    print(f"{label}: plotting {len(idx)} of {int(mask.sum())} (sorted by {SORT_BY})")
    if idx:
        plot_windows(idx, label)

### Detected peaks (p ≥ cut-off)

The candidates counted as peaks (`detected` in `candidates.csv`): probability ≥ cut-off, from a file that is not in the splits. Blue titles.

In [ ]:
show(d.detected.values, "detected peaks")

### Artifacts (p < cut-off)

The rejected candidates. Orange titles; near misses (0.5 ≤ p < cut-off) are the ones closest to the cut-off (`SORT_BY = "probability"` lists them first).

In [ ]:
show((d[P_COL] < mc.CUT).values, "artifacts")
n_split = int(((d[P_COL] >= mc.CUT) & d.file_in_splits).sum())
if n_split:
    print(f"({n_split} candidates with p >= cut-off are from files in the splits; they are in neither cell)")